# Train a free local model to replace Jev

This notebook fine-tunes small decision models on the dataset made by `python -m scanner.cli export-dataset`
and checks them against the held-out test set:

1. **Zero-shot Laya**: how far an untrained decision model gets.
2. **Fine-tuned Laya** (`convaiinnovations/laya`, ~0.4B parameters).
3. **Fine-tuned Qwen3.5-0.8B** with an Unsloth decision head.

It then compares them, suggests accept / reject thresholds for `config/stages.yaml`, and uploads the
best model as a GGUF file for llama.cpp's decision server.

**Before you start**

- Runtime > Change runtime type > **T4 GPU** (free).
- Settings come from your `.env`: drag it into the Files pane on the left, or add the same names as
  Colab secrets (key icon). It needs `HF_DATASET_REPO`, `HF_MODEL_REPO` and a Hugging Face token
  (`HF_TOKEN`) that can read the dataset and write the model repo.
- Expect roughly 30 to 60 minutes in total on a T4.
- **One topic per run.** For another topic, point `HF_DATASET_REPO` and `HF_MODEL_REPO` at that
  topic's repositories (in `.env` or as Colab secrets), and set `ACCEPT` / `REJECT` below to the
  thresholds of that topic's relevance stage in its config file.

Unsloth's decision-model API is new. If a cell fails, copy the error back to the conversation that
made this notebook.

In [ ]:
%%capture
!pip install --upgrade unsloth python-dotenv

## Settings

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path(".env"), override=True)  # the .env you uploaded; re-run this cell after editing it


def setting(name, default=None):
    # .env or environment first, then Colab secrets.
    if os.getenv(name):
        return os.environ[name].strip()
    try:
        from google.colab import userdata
        return userdata.get(name).strip()
    except Exception:  # no such secret, or its "Notebook access" is off
        return default


HF_DATASET_REPO = setting("HF_DATASET_REPO")
HF_MODEL_REPO = setting("HF_MODEL_REPO")
HF_TOKEN = setting("HF_TOKEN")
assert HF_DATASET_REPO, "Set HF_DATASET_REPO in .env or as a Colab secret"
assert HF_MODEL_REPO, "Set HF_MODEL_REPO in .env or as a Colab secret"

# Checked now so a missing token fails here, not after training when the model is uploaded.
from huggingface_hub import whoami

assert HF_TOKEN, (
    "Set HF_TOKEN (a Hugging Face token with write access, https://huggingface.co/settings/tokens) "
    "in .env or as a Colab secret, with 'Notebook access' switched on"
)
os.environ["HF_TOKEN"] = HF_TOKEN
account = whoami(token=HF_TOKEN)  # a 401 here means the token itself is wrong or revoked
access = account.get("auth", {}).get("accessToken", {})
print(f"logged in to Hugging Face as {account['name']}, token {access.get('displayName')!r} ({access.get('role')})")
assert access.get("role") != "read", "This token is read-only: create one with write access"

MODELS = {
    "laya": "convaiinnovations/laya",
    "qwen": "unsloth/Qwen3.5-0.8B",
}
# Relevance thresholds to score at: copy them from the relevance stage of the topic's config file.
ACCEPT, REJECT = 0.8, 0.2
# Longest input in tokens (question + title + abstract). Long questions need more than 512.
MAX_SEQ_LENGTH = {"laya": 1024, "qwen": 1024}
NEGATIVES_PER_POSITIVE = 10   # rejected papers trained on per relevant one
HARD_NEGATIVE_SHARE = 0.5     # of those, the share Jev found most borderline
EPOCHS = 2
SEED = 3407
PUBLIC_MODEL = True           # False uploads the trained model to a private repository
CHECK_NEGATIVES = 2000        # extra rejected papers to measure false accepts on
print(f"dataset {HF_DATASET_REPO} -> model {HF_MODEL_REPO}")

## Load the dataset

In [ ]:
import json

from datasets import load_dataset

dataset = load_dataset(HF_DATASET_REPO)
train_rows = [dict(row) for row in dataset["train"]]
test_rows = [dict(row) for row in dataset["test"]] if "test" in dataset else []
assert test_rows, "test.jsonl is empty: export with --test-size 300 and push again"

# The relevance question is the first one in every row (export-dataset writes it first).
RELEVANCE = next(iter(json.loads(train_rows[0]["questions"])))


def gold_relevant(row):
    return json.loads(row["gold"])[RELEVANCE] is True


def jev_probability(row):
    return json.loads(row["jev"]).get(RELEVANCE, {}).get("probability")


for name, rows in (("train", train_rows), ("test", test_rows)):
    relevant = sum(gold_relevant(row) for row in rows)
    print(f"{name}: {len(rows):,} papers, {relevant} relevant")

## A balanced training set

Fewer than 1% of papers are relevant, so a model trained on everything learns to say no. Train on
every relevant paper plus `NEGATIVES_PER_POSITIVE` rejected ones each: half of those are the rejected
papers Jev found most borderline (they teach where the line is), the rest are random.

In [ ]:
import random


def balanced(rows, negatives_per_positive, hard_share, seed):
    positives = [row for row in rows if gold_relevant(row)]
    negatives = [row for row in rows if not gold_relevant(row)]
    wanted = min(len(negatives), negatives_per_positive * len(positives))
    by_doubt = sorted(negatives, key=lambda row: jev_probability(row) or 0.0, reverse=True)
    hard = by_doubt[: int(wanted * hard_share)]
    rest = by_doubt[len(hard):]
    rng = random.Random(seed)
    chosen = positives + hard + rng.sample(rest, wanted - len(hard))
    rng.shuffle(chosen)
    return chosen


def one_question_each(rows):
    # The scanner asks one question per request. Qwen's Clef head reads all questions of a request
    # together, and only relevant papers have a sub-topic question, so rows holding both would let
    # the model tell relevance from the number of questions instead of from the abstract.
    split = []
    for row in rows:
        questions, gold = json.loads(row["questions"]), json.loads(row["gold"])
        for name in questions:
            split.append(row | {
                "questions": json.dumps({name: questions[name]}),
                "gold": json.dumps({name: gold[name]}),
            })
    return split


train_subset = balanced(train_rows, NEGATIVES_PER_POSITIVE, HARD_NEGATIVE_SHARE, SEED)
print(f"training on {len(train_subset):,} papers, {sum(map(gold_relevant, train_subset))} relevant")

# The test set is half relevant, but real scans are over 99% irrelevant, so a false-accept rate
# too small to show on 150 test papers still adds up over a year. Rejected papers the models never
# trained on measure it.
used = {row["id"] for row in train_subset}
unused_negatives = [row for row in train_rows if not gold_relevant(row) and row["id"] not in used]
check_negatives = random.Random(SEED).sample(unused_negatives, min(CHECK_NEGATIVES, len(unused_negatives)))
NEGATIVE_SHARE = sum(not gold_relevant(row) for row in train_rows) / len(train_rows)
print(f"{len(check_negatives):,} unseen rejected papers for measuring false accepts")

## How models are scored

Every test paper gets a relevance probability, split like the scanner does: at least `ACCEPT` is
accepted, at most `REJECT` is rejected, in between is "maybe" for you to review.

- **found**: share of relevant papers that are not rejected (accepted or maybe). Missed papers are
  the costly error, so this matters most.
- **precision**: share of accepted papers that are relevant.
- **maybe**: share of all papers left for you to review.
- **domain**: share of relevant papers whose sub-topic matches the gold answer.
- **false accepts**: share of unseen rejected papers the model accepts, and what that means per
  1,000 scanned papers at your real mix of relevant and irrelevant papers.

Test answers come from Jev unless you labelled the paper yourself, so on Jev-answered papers these
numbers measure agreement with Jev.

In [ ]:
def yes_probability(answer):
    # Unsloth answers a yes/no question with the Decision API's "noul" probability.
    if "noul" in answer:
        return float(answer["noul"])
    probabilities = answer.get("probabilities", {})
    for key in (True, "true", "True", "yes"):
        if key in probabilities:
            return float(probabilities[key])
    raise ValueError(f"no yes probability in {answer}")


def score(probabilities, rows, accept=ACCEPT, reject=REJECT, domains=None):
    relevant = [gold_relevant(row) for row in rows]
    accepted = [p >= accept for p in probabilities]
    rejected = [p <= reject for p in probabilities]
    n_relevant = sum(relevant)
    result = {
        "found": sum(r and not x for r, x in zip(relevant, rejected)) / max(1, n_relevant),
        "precision": sum(r and a for r, a in zip(relevant, accepted)) / max(1, sum(accepted)),
        "maybe": sum(not a and not x for a, x in zip(accepted, rejected)) / len(rows),
    }
    if domains is not None:
        pairs = [(d, json.loads(row["gold"])) for d, row in zip(domains, rows) if gold_relevant(row)]
        pairs = [(d, gold) for d, gold in pairs if len(gold) > 1]
        hits = [d == next(v["label"] for k, v in gold.items() if k != RELEVANCE) for d, gold in pairs]
        result["domain"] = sum(hits) / len(hits) if hits else None
    return result


def false_accepts(negative_probabilities, accept=ACCEPT):
    rate = sum(p >= accept for p in negative_probabilities) / max(1, len(negative_probabilities))
    return {"false accepts": rate, "per 1000 scanned": rate * NEGATIVE_SHARE * 1000}


def show(name, result):
    parts = [
        f"{key} {value:.1f}" if key == "per 1000 scanned" else f"{key} {value:.1%}"
        for key, value in result.items() if value is not None
    ]
    print(f"{name:<30}" + "   ".join(parts))


results = {}
human_test = [row for row in test_rows if row["label_source"] == "human"]
if human_test:
    show("Jev (your labels)", score([jev_probability(row) or 0.0 for row in human_test], human_test))
print(f"{len(test_rows)} test papers, {len(human_test)} with your own labels")

In [ ]:
import torch
from unsloth import DecisionTrainer, FastDecisionModel
from transformers import TrainingArguments


def predict_all(model, tokenizer, rows):
    # One question per request, like a scan: relevance alone, then the sub-topic on its own.
    FastDecisionModel.for_inference(model)
    probabilities, domains = [], []
    for row in rows:
        questions = json.loads(row["questions"])
        answer = FastDecisionModel.predict(model, tokenizer, row["state"], {RELEVANCE: questions[RELEVANCE]})
        probabilities.append(yes_probability(answer[RELEVANCE]))
        others = [name for name in questions if name != RELEVANCE]
        if others:
            name = others[0]
            answer = FastDecisionModel.predict(model, tokenizer, row["state"], {name: questions[name]})
            domains.append(answer[name].get("answer"))
        else:
            domains.append(None)
    return probabilities, domains


def evaluate(name, model, tokenizer):
    probabilities, domains = predict_all(model, tokenizer, test_rows)
    negatives, _ = predict_all(model, tokenizer, check_negatives)
    results[name] = (probabilities, domains, negatives)
    show(name, score(probabilities, test_rows, domains=domains) | false_accepts(negatives))


def free_gpu():
    import gc
    gc.collect()
    torch.cuda.empty_cache()

## 1. Zero-shot Laya

In [ ]:
model, tokenizer = FastDecisionModel.from_pretrained(MODELS["laya"], max_seq_length=MAX_SEQ_LENGTH["laya"])
evaluate("laya zero-shot", model, tokenizer)
del model
free_gpu()

## 2 and 3. Fine-tune Laya and Qwen

The same recipe for both, after Unsloth's `train_decision_from_lm.py`: LoRA, a few epochs on the
balanced set, then temperature calibration on a 10% holdout of it. Each trained model is saved to
`/content/models/<name>`.

In [ ]:
def fine_tune(name, max_seq_length):
    model, tokenizer = FastDecisionModel.from_pretrained(
        MODELS[name], max_seq_length=max_seq_length, random_state=SEED,
    )
    items, report = FastDecisionModel.build_dataset(one_question_each(train_subset), tokenizer, model)
    print(f"{len(items):,} training decisions, {report['skipped']} skipped {report['reason'] or ''}")
    train, holdout = FastDecisionModel.split_holdout(items, SEED)

    model = FastDecisionModel.get_peft_model(model, r=32, lora_alpha=32, random_state=SEED)
    batch, accumulate = 8, 2
    steps = EPOCHS * -(-len(train) // (batch * accumulate))
    trainer = DecisionTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train,
        head_learning_rate=3e-4,
        args=TrainingArguments(
            output_dir=f"outputs/{name}",
            per_device_train_batch_size=batch,
            gradient_accumulation_steps=accumulate,
            learning_rate=1e-4,
            lr_scheduler_type="cosine",
            warmup_steps=max(1, steps // 20),
            weight_decay=0.01,
            num_train_epochs=EPOCHS,
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            report_to="none",
            save_strategy="no",
            seed=SEED,
        ),
    )
    trainer.train()
    if holdout:
        calibration = FastDecisionModel.calibrate(model, tokenizer, holdout)
        print({key: calibration[key] for key in ("accuracy", "ece") if key in calibration})

    evaluate(f"{name} fine-tuned", model, tokenizer)
    model.save_pretrained_merged(f"/content/models/{name}")
    del model, trainer
    free_gpu()

In [ ]:
fine_tune("laya", max_seq_length=MAX_SEQ_LENGTH["laya"])

In [ ]:
fine_tune("qwen", max_seq_length=MAX_SEQ_LENGTH["qwen"])

## Compare

Scores at `ACCEPT` / `REJECT` from the settings, then, for each fine-tuned model, what other
thresholds would change:

- **reject**: lower means fewer missed papers but more "maybe" papers to review. "Review per 1000"
  estimates how many papers per 1,000 scanned would land in "maybe", at your real share of
  irrelevant papers.
- **accept**: lower means fewer "maybe" papers but more irrelevant ones accepted.

Pick the values from the table rather than optimising a single number: the test set holds only 150
relevant papers, so a difference of one paper is noise. Put them into the relevance stage of
`config/stages.yaml` when you switch the scanner to this model.

In [ ]:
def review_per_1000(probabilities, negatives, accept, reject):
    # "maybe" papers per 1,000 scanned, from the relevant test papers and the unseen rejected ones.
    relevant = [p for p, row in zip(probabilities, test_rows) if gold_relevant(row)]
    def maybe_share(values):
        return sum(reject < p < accept for p in values) / max(1, len(values))
    return 1000 * ((1 - NEGATIVE_SHARE) * maybe_share(relevant) + NEGATIVE_SHARE * maybe_share(negatives))


def threshold_table(name, probabilities, negatives):
    print(f"\n{name}")
    print(f"   reject <=    found   review per 1000   (accept {ACCEPT})")
    for reject in sorted({r for r in (0.5, 0.3, 0.2, 0.1, 0.05, 0.02, REJECT) if r < ACCEPT}, reverse=True):
        found = score(probabilities, test_rows, ACCEPT, reject)["found"]
        print(f"   {reject:9.2f}   {found:6.1%}   {review_per_1000(probabilities, negatives, ACCEPT, reject):15.1f}")
    print(f"   accept >=   precision   false accepts per 1000   review per 1000   (reject {REJECT})")
    for accept in sorted({a for a in (0.3, 0.4, 0.5, 0.7, 0.8, 0.9, ACCEPT) if a > REJECT}):
        precision = score(probabilities, test_rows, accept, REJECT)["precision"]
        wrong = false_accepts(negatives, accept)["per 1000 scanned"]
        print(f"   {accept:9.2f}   {precision:9.1%}   {wrong:22.1f}   {review_per_1000(probabilities, negatives, accept, REJECT):15.1f}")


best, best_key = None, None
for name, (probabilities, domains, negatives) in results.items():
    result = score(probabilities, test_rows, domains=domains) | false_accepts(negatives)
    show(name, result)
    key = (result["found"], -result["false accepts"], result["precision"], -result["maybe"])
    if name.endswith("fine-tuned") and (best_key is None or key > best_key):
        best, best_key = name.split()[0], key
for name, (probabilities, domains, negatives) in results.items():
    if name.endswith("fine-tuned"):
        threshold_table(name, probabilities, negatives)
print(f"\nbest at accept {ACCEPT} / reject {REJECT}: {best}")

## Upload the best model as GGUF

Change `best` above if you prefer the other model. This converts it for llama.cpp's decision server
into `/content/gguf` and uploads it to `HF_MODEL_REPO` as `<model>-q8_0.gguf`, for example
`qwen-q8_0.gguf` (public unless `PUBLIC_MODEL = False`). If the upload fails, fix the cause and run
only the upload cell again; the conversion is kept.

The conversion uses Unsloth's GGUF export, which also writes the calibration into the file. If that
fails (in Colab it once failed with `No module named 'gguf.scripts'`), it falls back to the converter
of llama.cpp release b11443, the first one with decision models, together with that release's own
`gguf` package. Check a fallback model with `python -m scanner.cli evaluate --backend local`: its
numbers should match the ones above.

In [ ]:
import shutil
import subprocess
import sys

shutil.rmtree("/content/gguf", ignore_errors=True)  # never upload leftovers of an earlier run
model, tokenizer = FastDecisionModel.from_pretrained(f"/content/models/{best}")
try:
    model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q8_0")
except Exception as error:
    print(f"Unsloth's GGUF export failed ({error!r}), converting with llama.cpp b11443 instead")
    shutil.rmtree("/content/gguf", ignore_errors=True)
    Path("/content/gguf").mkdir()
    merged = f"/content/models/{best}_merged"
    model.save_pretrained_merged(merged, tokenizer)
    llama = Path("/content/llama.cpp-b11443")
    if not llama.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "b11443",
                        "https://github.com/ggml-org/llama.cpp", str(llama)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(llama / "gguf-py")], check=True)
    subprocess.run([sys.executable, str(llama / "convert_hf_to_gguf.py"), merged,
                    "--outfile", f"/content/gguf/{best}-q8_0.gguf", "--outtype", "q8_0"], check=True)

# The image encoder (mmproj) is not needed: papers are text only.
GGUF_FILE = next(p for p in sorted(Path("/content/gguf").rglob("*.gguf")) if not p.name.startswith("mmproj"))
print(f"{GGUF_FILE} ({GGUF_FILE.stat().st_size / 1e6:.0f} MB)")

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_MODEL_REPO, private=not PUBLIC_MODEL, exist_ok=True)
api.upload_file(
    path_or_fileobj=str(GGUF_FILE),
    path_in_repo=f"{best}-q8_0.gguf",
    repo_id=HF_MODEL_REPO,
    commit_message=f"{best} decision model, trained with one question per request",
)
print(f"uploaded https://huggingface.co/{HF_MODEL_REPO}/blob/main/{best}-q8_0.gguf")
print(f"locally: hf download {HF_MODEL_REPO} {best}-q8_0.gguf --local-dir models/event-cameras")